# D2c 전이학습과 데이터 증강 — 실습 (W6, D2 3부작 완결)

> Colab에서 **런타임 → GPU** 권장(CPU도 동작 — 전이학습 셀은 몇 분 걸립니다).
> 위에서부터 한 셀씩 `Shift+Enter`로 실행하세요. `___` 빈칸은 직접 채웁니다.

**이 실습이 끝나면**
1. 사전학습 ResNet18을 로드해 **동결 → fc 교체** 4단계를 수행한다
2. 학습 대상이 **5,130개(전체의 0.05%)** 가 됨을 검산한다
3. 데이터 증강 6종을 눈으로 확인한다 (train에만!)
4. **fc만 2 epoch 학습**해 SmallCNN(45.3%)과 최종 대결한다
5. D2 3부작 스코어보드를 완성한다

**7단계 멘탈모델 초점:** 모델 + 활용

## Part A. 사전학습 ResNet18 로드 — 남이 만든 눈 구경
ImageNet 128만 장으로 학습된 가중치가 자동 다운로드됩니다(1회, ~45MB).

In [ ]:
import torch                                            # PyTorch
import torch.nn as nn                                   # 신경망 모듈
import matplotlib.pyplot as plt                         # 그래프
from torchvision import datasets, transforms, models    # 데이터·변환·사전학습 모델
from torch.utils.data import DataLoader, Subset         # 로더·부분집합

device = 'cuda' if torch.cuda.is_available() else 'cpu' # 장치
torch.manual_seed(0)                                    # 재현성

resnet = models.resnet18(weights='IMAGENET1K_V1')       # ① 로드 — 이미 1000클래스 분류기
print('마지막 층:', resnet.fc)                          # Linear(512, 1000) — 여기만 손댄다
n_total = sum(p.numel() for p in resnet.parameters())   # 전체 파라미터(D1c numel)
print('전체 파라미터:', f'{n_total:,}')                 # ~11.7M — SmallCNN(268,650)의 44배

## Part B. 동결 → 교체 ⭐ — 순서가 생명
**②동결을 먼저, ③교체를 나중에.** 새로 만든 층은 requires_grad=True로 태어나므로, 이 순서면 자동으로 "새 fc만 학습 대상"이 됩니다.

In [ ]:
for p in resnet.parameters():                           # ② 동결 — 모든 파라미터의
    p.requires_grad = ___                               # ✍️ 빈칸: autograd 스위치 끄기(D1b!)

resnet.fc = nn.Linear(resnet.fc.___, 10)                # ✍️ 빈칸: 입력 특징 수 속성(하드코딩 금지)
resnet = resnet.to(device)                              # 장치로

n_train = sum(p.numel() for p in resnet.parameters() if p.requires_grad)  # 학습 대상만
n_all = sum(p.numel() for p in resnet.parameters())     # 전체
print('전체:', f'{n_all:,}', '| 학습 대상:', f'{n_train:,}',
      f'({100 * n_train / n_all:.2f}%)')                # 5,130개 = 0.05%!
print('학습되는 층:', [n for n, p in resnet.named_parameters() if p.requires_grad])  # fc뿐

> 1,118만 개짜리 모델에서 **5,130개(512×10+10)만 학습** — 그래서 적은 데이터·짧은 시간으로 됩니다. 학습 자유도가 작아 과적합 위험도 급감(M2).

## Part C. 데이터 증강 — 공짜 데이터를 눈으로
정답이 보존되는 변형(뒤집기·어긋난 자르기)으로 사실상 새로운 학습 샘플을 만듭니다. **train에만!**

In [ ]:
raw_ds = datasets.CIFAR10('./data', train=True, download=True)  # 변환 없이(PIL 이미지)
img_pil, label = raw_ds[7]                              # 말(horse) 사진 — D2a의 그 말

aug = transforms.Compose([                              # 증강 파이프라인(학습용)
    transforms.___(),                                   # ✍️ 빈칸: 좌우 뒤집기(확률 0.5) 변환
    transforms.RandomCrop(32, padding=___),             # ✍️ 빈칸: 4픽셀 테두리 후 어긋난 자르기
])

fig, axes = plt.subplots(1, 6, figsize=(12, 2.4))       # 같은 사진 6가지 변형
axes[0].imshow(img_pil); axes[0].set_title('original')  # 원본
for i, ax in enumerate(axes[1:], 1):
    ax.imshow(aug(img_pil))                             # 매번 다른 무작위 변형
    ax.set_title(f'augmented {i}')                      # 제목(영어)
for ax in axes: ax.axis('off')                          # 축 끄기
plt.show()                                              # 전부 '말'이라는 정답은 그대로!

## Part D. 입력 규격 맞추기 — Resize(224)
ResNet18은 224×224로 학습된 눈. CIFAR 32×32를 키워서 규격을 맞춥니다. **평가용 변환에는 증강이 없습니다.**

In [ ]:
tf224 = transforms.Compose([                            # 전이학습용 파이프라인
    transforms.Resize(___),                             # ✍️ 빈칸: ResNet 입력 규격(한 변)
    transforms.ToTensor(),                              # 0~1 텐서
])
train224 = datasets.CIFAR10('./data', train=True,  transform=tf224)  # 학습(이번 실습은 증강 없이 단순화)
test224  = datasets.CIFAR10('./data', train=False, transform=tf224)  # 평가(항상 원본 규격)
train_loader = DataLoader(Subset(train224, range(2000)), batch_size=32, shuffle=True)  # 겨우 2천 장
test_loader  = DataLoader(Subset(test224,  range(1000)), batch_size=64)                # 평가 1천 장
xb, yb = next(iter(train_loader))                       # 배치 확인
print('배치 shape:', xb.shape)                          # (32,3,224,224) — 규격 맞춤 완료

## Part E. fc만 학습 → 최종 대결 ⭐⭐
학습 루프는 이번에도 D1c 그대로. 다른 점 하나 — **옵티마이저에 fc의 파라미터만** 넘깁니다(이중 안전장치).

In [ ]:
criterion = nn.CrossEntropyLoss()                       # 분류 손실(D1b)
optimizer = torch.optim.Adam(resnet.fc.___(), lr=1e-3)  # ✍️ 빈칸: fc의 파라미터만 넘기는 메서드

for epoch in range(2):                                  # 겨우 2 epoch
    resnet.train()                                      # 학습 모드
    running = 0.0
    for xb, yb in train_loader:                         # D1c 5단계 그대로
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(resnet(xb), yb)
        loss.backward()                                 # 동결된 층은 건너뜀(빠름)
        optimizer.step()
        running += loss.item()
    print(f'epoch {epoch+1}: train loss = {running / len(train_loader):.4f}')

resnet.___()                                            # ✍️ 빈칸: 평가 모드 전환(D1c)
correct = total = 0
with torch.no_grad():                                   # 기록 끄기
    for xb, yb in test_loader:
        pred = resnet(xb.to(device)).argmax(1).cpu()    # 예측
        correct += (pred == yb).sum().item(); total += yb.size(0)
acc = correct / total
print('전이학습 test accuracy:', round(acc, 3))         # SmallCNN(0.453)을 넘는가?

In [ ]:
print('===== D2 3부작 스코어보드 (CIFAR-10) =====')      # 3주 서사 완성
print('무작위 찍기            : 0.100        (기준선)')
print('MLP  (157만 par, 1만장): 0.335        (D2b — 공간 구조 모름)')
print('CNN  ( 27만 par, 1만장): 0.453        (D2b — 구조가 용량을 이김)')
print(f'전이학습(5,130 par 학습, 2천장): {acc:.3f}    (D2c — 좋은 초기값이 밑바닥을 이김)')
print()
print('학습 데이터는 1/5, 학습 파라미터는 1/50인데 성능은 최고 —')
print('"빌린 눈"의 위력. 실무 영상 인식이 전이학습에서 시작하는 이유.')

## 🤖 AI 코파일럿 활용 (선택) — ai-native v1
막히면 AI 튜터에게 묻되, **먼저 스스로 생각**하고 답을 **실행으로 검증**하세요.

**좋은 질문 예시**
- "동결과 교체의 순서를 바꾸면 무슨 일이 생기는지 내가 예측할 테니 확인해 줘." (그리고 직접 실험!)
- "MNIST에 좌우 뒤집기 증강을 쓰면 왜 위험한지 내 설명을 채점해 줘."
- "resnet50으로 바꾸면 fc의 in_features가 몇인지 확인하는 코드를 내가 짜 볼게." (답: 2048)
- "파인튜닝을 lr=1e-3으로 전체에 하면 왜 위험한지 D1b의 '세 운명'으로 설명해 볼게."

**가드레일**
1. 먼저 손으로 생각 → 그 다음 AI
2. AI 코드는 *왜 그런지* 설명할 수 있을 때만 사용
3. AI 출력은 실행으로 검증

## 정리 & 자가 점검 — D2 3부작 완결 🎉

**오늘 한 일 3줄**
1. ResNet18을 로드 → 동결 → fc 교체 — 학습 대상 5,130개(0.05%) 확인
2. 증강 6종을 눈으로 확인(정답 보존 변형, train에만)
3. 2천 장·2 epoch·fc만으로 SmallCNN을 크게 추월 — **좋은 초기값이 밑바닥 학습을 이긴다**

**스스로 점검**
- [ ] 동결→교체 순서를 바꾸면 생기는 일을 설명할 수 있다
- [ ] 5,130 = 512×10+10 계산을 할 수 있다
- [ ] Resize(224)가 왜 필요한지 안다
- [ ] 증강을 test에 쓰면 안 되는 이유를 M2로 설명할 수 있다
- [ ] D2 3부작 서사(부품→조립→빌리기)를 스코어보드와 함께 말할 수 있다

**🔹심화 (선택)**
- **순서 실험:** 교체→동결 순서로 바꿔 학습 대상 파라미터 수를 확인해 보세요(0개가 됨 — 학습 불가).
- **증강 추가 학습:** Part D의 tf224 앞에 `RandomHorizontalFlip()`을 넣고 재학습 — 2 epoch 짧은 학습에서도 차이가 나는지.
- **부분 파인튜닝:** `for p in resnet.layer4.parameters(): p.requires_grad = True` 후 옵티마이저에 두 그룹(layer4는 lr=1e-5, fc는 1e-3)을 넘겨 비교.
- Colab GPU에서 전체 데이터(5만 장)·5 epoch로 키워 보세요 — 80%대 진입을 확인할 수 있습니다.